# NYC Taxi Trip Analytics — ETL Orchestration

Replaces the exploratory `simple_ETL.ipynb` / `chunked_ETL.ipynb` prototypes.

**What this notebook does.** Streams the NYC Yellow Taxi CSV in chunks, validates
each chunk against the schema contract, derives the time features, applies the
business-rule filters, and appends to the `nyc_raw_taxi` / `nyc_clean_taxi`
PostgreSQL tables. All of that logic lives in `src/nyc_taxi/etl.py`; this notebook
only composes and reports.

**Prerequisites**
1. `pip install -e ".[notebooks,data]"`
2. A running PostgreSQL instance and `POSTGRES_PASSWORD` exported (see `.env.example`).
3. `sql/00_init_database.sql` executed once (DataGrip).

Set `NYC_TAXI_MAX_ROWS` to cap ingestion when iterating against the 12.7M-row month.

**Headless equivalent:** `nyc-taxi-etl`

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from __future__ import annotations

import pandas as pd
from dotenv import load_dotenv

from nyc_taxi.config import load_config
from nyc_taxi.database import build_engine, read_table
from nyc_taxi.etl import TaxiETLPipeline
from nyc_taxi.utils import setup_logging

load_dotenv()
setup_logging()

---
## 1. Configuration

Chunk size, schema contract and every filter bound come from `configs/train.yaml`.

In [ ]:
cfg = load_config()

print(f"Chunk size     : {cfg.etl.chunk_size:,} rows")
print(f"Raw table      : {cfg.database.raw_table}")
print(f"Clean table    : {cfg.database.clean_table}")
print(f"Target host    : {cfg.database.host}:{cfg.database.port}/{cfg.database.database}")

In [ ]:
pd.DataFrame(
    [{"rule": key, "value": value} for key, value in cfg.etl.filters.items()]
).style.hide(axis="index").set_caption("Business rules applied during transform")

---
## 2. Source data

`resolve_source_path` resolves the input in order: `etl.source.local_path`
(env `NYC_TAXI_SOURCE_PATH`), then the NYC TLC public parquet URL, then the
Kaggle mirror via `kagglehub`. None of these require credentials.

In [ ]:
pipeline = TaxiETLPipeline(cfg)
source_path = pipeline.resolve_source_path()

print(f"Source file : {source_path}")
print(f"Format      : {pipeline._resolve_format(source_path)}")
print(f"Size        : {source_path.stat().st_size / 1e9:.2f} GB")

In [ ]:
preview_chunk = next(pipeline.extract(source_path))
preview_chunk.head()

---
## 3. Run the pipeline

A single call. Progress is reported per chunk through the configured logger.

In [ ]:
stats = pipeline.run()

pd.DataFrame([stats.as_dict()]).T.rename(columns={0: "value"})

---
## 4. Verify the load

The counts below must agree with `sql/02_analytics.sql`, query 1.

In [ ]:
engine = build_engine(cfg)
try:
    clean_sample = read_table(engine, cfg.database.clean_table, limit=5)
finally:
    engine.dispose()

clean_sample

---
## 5. Next step

Apply the constraints and indexes, then train:

```bash
psql "$DATABASE_URL" -f sql/01_schema.sql   # or run it from DataGrip
jupyter lab notebooks/01_baseline_training.ipynb
```